In [ ]:
# Merge datasets, subsample and split

import pandas as pd
from sklearn.model_selection import train_test_split

# Step 1: Load Spanish-English
with open("europarl-v7.es-en.en", encoding="utf-8") as f_en_es, open("europarl-v7.es-en.es", encoding="utf-8") as f_es:
    en_es_sentences = [line.strip() for line in f_en_es]
    es_sentences = [line.strip() for line in f_es]

df_es_en = pd.DataFrame({
    "en": en_es_sentences,
    "es": es_sentences
})

# Step 2: Load Swedish-English
with open("europarl-v7.sv-en.en", encoding="utf-8") as f_en_sv, open("europarl-v7.sv-en.sv", encoding="utf-8") as f_sv:
    en_sv_sentences = [line.strip() for line in f_en_sv]
    sv_sentences = [line.strip() for line in f_sv]

df_sv_en = pd.DataFrame({
    "en": en_sv_sentences,
    "sv": sv_sentences
})

# Step 3: Inner join on English text
df_pivot = pd.merge(df_es_en, df_sv_en, on="en", how="inner")

# Optional: sample to reduce size (e.g. 1%)
df_pivot = df_pivot.sample(frac=0.0003, random_state=42).reset_index(drop=True)

# Step 4: Train/val/test split
df_trainval, df_test = train_test_split(df_pivot, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_trainval, test_size=0.125, random_state=42)  # 10% of 80%

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

Train: 16569 | Val: 2367 | Test: 4735


In [ ]:
# Average BLEU

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import pandas as pd

# ---------------- Configuration ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 32
BATCH_SIZE = 16
EMBED_DIM = 128
HIDDEN_DIM = 256
DROPOUT = 0.1
NUM_EPOCHS = 10
LR = 3e-4

# ---------------- Tokenizer & Model ----------------
bert_model = BertModel.from_pretrained("google/bert_uncased_L-4_H-256_A-4").to(DEVICE)
tokenizer = BertTokenizer.from_pretrained("google/bert_uncased_L-4_H-256_A-4")

# ---------------- Dataset Definition ----------------
class TranslationDataset(Dataset):
    def __init__(self, df, src_col, tgt_col, tokenizer, max_len):
        self.src_sentences = df[src_col].tolist()
        self.tgt_sentences = df[tgt_col].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        src = self.src_sentences[idx]
        tgt = self.tgt_sentences[idx]
        src_enc = self.tokenizer(src, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        tgt_enc = self.tokenizer(tgt, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
        return src_enc["input_ids"].squeeze(0), src_enc["attention_mask"].squeeze(0), tgt_enc["input_ids"].squeeze(0)

def collate_fn(batch):
    src_ids, src_mask, tgt_ids = zip(*batch)
    return torch.stack(src_ids), torch.stack(src_mask), torch.stack(tgt_ids)

# ---------------- Model Components ----------------
class BertEncoder(nn.Module):
    def __init__(self, bert_model, hidden_size):
        super().__init__()
        self.bert = bert_model
        self.proj = nn.Linear(bert_model.config.hidden_size, hidden_size)

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        projected = self.proj(cls_output)
        return projected.unsqueeze(0)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, hidden):
        embedded = self.dropout(self.embedding(tgt))
        output, hidden = self.rnn(embedded, hidden)
        return self.fc(output), hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src_ids, src_mask, tgt):
        hidden = self.encoder(src_ids, src_mask)
        output, _ = self.decoder(tgt[:, :-1], hidden)
        return output

# ---------------- Load and Merge Data ----------------
df_es_en = pd.DataFrame({
    "en": [line.strip() for line in open("europarl-v7.es-en.en", encoding="utf-8")],
    "es": [line.strip() for line in open("europarl-v7.es-en.es", encoding="utf-8")]
})
df_sv_en = pd.DataFrame({
    "en": [line.strip() for line in open("europarl-v7.sv-en.en", encoding="utf-8")],
    "sv": [line.strip() for line in open("europarl-v7.sv-en.sv", encoding="utf-8")]
})
df_all = pd.merge(df_es_en, df_sv_en, on="en", how="inner").sample(frac=0.01, random_state=42).reset_index(drop=True)
df_trainval, df_test = train_test_split(df_all, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_trainval, test_size=0.125, random_state=42)

# ---------------- Spanish → English ----------------
train_loader = DataLoader(TranslationDataset(df_train, "es", "en", tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(TranslationDataset(df_val, "es", "en", tokenizer, MAX_LEN), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader = DataLoader(TranslationDataset(df_test, "es", "en", tokenizer, MAX_LEN), batch_size=BATCH_SIZE, collate_fn=collate_fn)

encoder_es_en = BertEncoder(bert_model, HIDDEN_DIM)
decoder_es_en = Decoder(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM, DROPOUT)
model_es_en = Seq2Seq(encoder_es_en, decoder_es_en).to(DEVICE)
optimizer_es_en = optim.Adam(model_es_en.decoder.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

# ---------------- English → Swedish ----------------
train_loader_sv = DataLoader(TranslationDataset(df_train, "en", "sv", tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader_sv = DataLoader(TranslationDataset(df_val, "en", "sv", tokenizer, MAX_LEN), batch_size=BATCH_SIZE, collate_fn=collate_fn)
test_loader_sv = DataLoader(TranslationDataset(df_test, "en", "sv", tokenizer, MAX_LEN), batch_size=BATCH_SIZE, collate_fn=collate_fn)

encoder_en_sv = BertEncoder(bert_model, HIDDEN_DIM)
decoder_en_sv = Decoder(tokenizer.vocab_size, EMBED_DIM, HIDDEN_DIM, DROPOUT)
model_en_sv = Seq2Seq(encoder_en_sv, decoder_en_sv).to(DEVICE)
optimizer_en_sv = optim.Adam(model_en_sv.decoder.parameters(), lr=LR)

# ---------------- Training & Eval ----------------
def train(model, loader, optimizer):
    model.train()
    total_loss = 0
    for src_ids, src_mask, tgt_ids in tqdm(loader):
        src_ids, src_mask, tgt_ids = src_ids.to(DEVICE), src_mask.to(DEVICE), tgt_ids.to(DEVICE)
        output = model(src_ids, src_mask, tgt_ids)
        loss = criterion(output.reshape(-1, tokenizer.vocab_size), tgt_ids[:, 1:].reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src_ids, src_mask, tgt_ids in loader:
            src_ids, src_mask, tgt_ids = src_ids.to(DEVICE), src_mask.to(DEVICE), tgt_ids.to(DEVICE)
            output = model(src_ids, src_mask, tgt_ids)
            loss = criterion(output.reshape(-1, tokenizer.vocab_size), tgt_ids[:, 1:].reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

# Train both models
for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}")
    loss_es_en = train(model_es_en, train_loader, optimizer_es_en)
    val_es_en = evaluate(model_es_en, val_loader)
    print(f"ES→EN Train Loss: {loss_es_en:.4f} | Val Loss: {val_es_en:.4f}")

    loss_en_sv = train(model_en_sv, train_loader_sv, optimizer_en_sv)
    val_en_sv = evaluate(model_en_sv, val_loader_sv)
    print(f"EN→SV Train Loss: {loss_en_sv:.4f} | Val Loss: {val_en_sv:.4f}")

# Final test losses
test_loss_es_en = evaluate(model_es_en, test_loader)
test_loss_en_sv = evaluate(model_en_sv, test_loader_sv)
print(f"\nTest Loss ES→EN: {test_loss_es_en:.4f}")
print(f"Test Loss EN→SV: {test_loss_en_sv:.4f}")

# ---------------- Inference: Spanish → English → Swedish ----------------
def translate_pipeline(spanish_text):
    model_es_en.eval()
    model_en_sv.eval()

    # Step 1: Spanish → English
    enc = tokenizer(spanish_text, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LEN).to(DEVICE)
    with torch.no_grad():
        hidden_en = model_es_en.encoder(enc["input_ids"], enc["attention_mask"])
        input_en = torch.tensor([[tokenizer.cls_token_id]], device=DEVICE)
        output_en = []
        for _ in range(MAX_LEN):
            logits, hidden_en = model_es_en.decoder(input_en, hidden_en)
            next_token = logits.argmax(-1)[:, -1:]
            output_en.append(next_token.item())
            input_en = next_token
            if next_token.item() == tokenizer.sep_token_id:
                break
        english_text = tokenizer.decode(output_en, skip_special_tokens=True)

    # Step 2: English → Swedish
    enc_sv = tokenizer(english_text, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LEN).to(DEVICE)
    with torch.no_grad():
        hidden_sv = model_en_sv.encoder(enc_sv["input_ids"], enc_sv["attention_mask"])
        input_sv = torch.tensor([[tokenizer.cls_token_id]], device=DEVICE)
        output_sv = []
        for _ in range(MAX_LEN):
            logits, hidden_sv = model_en_sv.decoder(input_sv, hidden_sv)
            next_token = logits.argmax(-1)[:, -1:]
            output_sv.append(next_token.item())
            input_sv = next_token
            if next_token.item() == tokenizer.sep_token_id:
                break
        swedish_text = tokenizer.decode(output_sv, skip_special_tokens=True)

    return english_text, swedish_text

# Example usage
es_example = "¿Dónde está la estación de tren?"
en_out, sv_out = translate_pipeline(es_example)
print(f"\nES: {es_example}")
print(f"EN: {en_out}")
print(f"SV: {sv_out}")


Epoch 1


100%|██████████| 34521/34521 [10:29<00:00, 54.84it/s]


ES→EN Train Loss: 0.7476 | Val Loss: 0.6507


100%|██████████| 34521/34521 [10:22<00:00, 55.44it/s]


EN→SV Train Loss: 1.1067 | Val Loss: 0.8532

Epoch 2


100%|██████████| 34521/34521 [10:26<00:00, 55.11it/s]


ES→EN Train Loss: 0.6275 | Val Loss: 0.6256


 46%|████▌     | 15923/34521 [04:47<05:38, 54.99it/s]

In [ ]:
# Calculate average BLEU

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
smoothie = SmoothingFunction().method4

# Collect predictions and references
references = []
predictions = []

for es_sent, sv_ref in tqdm(zip(df_test["es"], df_test["sv"]), total=len(df_test)):
    _, sv_pred = translate_pipeline(es_sent)

    # Tokenize reference and prediction
    ref_tokens = tokenizer.tokenize(sv_ref)
    pred_tokens = tokenizer.tokenize(sv_pred)

    references.append([ref_tokens])  # list of references (single ref here)
    predictions.append(pred_tokens)

# Compute average BLEU-4
bleu_scores = [sentence_bleu(ref, pred, smoothing_function=smoothie) for ref, pred in zip(references, predictions)]
avg_bleu = sum(bleu_scores) / len(bleu_scores)

print(f"\nAverage BLEU score on test set: {avg_bleu:.4f}")